# 🏷️ Label Semantics Analysis: name_only/L2/L3

**Experiment — Label Semantics Analysis using Generated Descriptions**

- **name_only:** Sadece label ismi ("world", "sports", ...)
- **L2 (LLM single):** Task-aware tek açıklama (12-25 kelime)
- **L3 (LLM multi):** Task-aware 3 açıklama (mean pooling)

**Total:** 27 experiments (9 datasets × 1 model × 3 modes)

**Config Location:** `experiments/`

## 🚀 Setup

In [ ]:
# Clone from GitHub (RECOMMENDED)
!git clone https://github.com/EngindalgaMaku/zeroshot-text-classification-benchmark-.git
%cd zeroshot-text-classification-benchmark-
!pip install -q -r requirements.txt

# Verify config directory exists
!ls -la experiments/exp_*_mpnet_*.yaml | head -30

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## ⚙️ Configuration

In [ ]:
# ⚙️ SETTINGS
SKIP_EXISTING = True  # Skip if results already exist
SHOW_PROGRESS = True  # Show progress after each group

# You can filter by dataset or model to run subset
FILTER_DATASET = None  # e.g., "ag_news" or None for all
FILTER_MODEL = None    # e.g., "all-mpnet-base-v2" or None for all

# NOTE: jina model has transformers compatibility issues in Colab
# Skip it for now: use FILTER_MODEL to run specific models

print(f"Skip existing: {SKIP_EXISTING}")
print(f"Show progress: {SHOW_PROGRESS}")
print(f"Filter dataset: {FILTER_DATASET or 'All'}")
print(f"Filter model: {FILTER_MODEL or 'All (except jina - compatibility issues)'}")

## 🔧 Helper Functions

In [ ]:
import json
import subprocess
from pathlib import Path
from IPython.display import display, HTML

def run_experiment(config_path, skip_existing=True):
    """Run single experiment."""
    if config_path is None:
        print("❌ FAILED: config_path is None")
        return None
    
    # Convert to absolute path string
    config_str = str(config_path.resolve())
    cmd = ["python", "main.py", "--config", config_str]
    
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"❌ FAILED: {config_path.name}")
        print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
        return None
    
    # Load metrics from new output location
    exp_name = config_path.stem
    metrics_file = Path("results/raw/label_semantics") / f"{exp_name}_metrics.json"
    
    if metrics_file.exists():
        with open(metrics_file) as f:
            return json.load(f)
    return None

def show_comparison(l1, l2, l3, dataset, model):
    """Show name_only/L2/L3 comparison."""
    html = f"<h3>{dataset} - {model}</h3><table border='1' style='border-collapse: collapse; font-family: monospace;'>"
    html += "<tr><th>Mode</th><th>Accuracy</th><th>Macro F1</th><th>Gain vs name_only</th></tr>"
    
    if l1:
        html += f"<tr><td><b>name_only</b></td><td>{l1['accuracy']:.4f}</td><td>{l1['macro_f1']:.4f}</td><td>-</td></tr>"
    else:
        html += "<tr><td><b>name_only</b></td><td colspan='3'>❌ FAILED</td></tr>"
    
    if l2:
        gain = f"+{(l2['accuracy'] - l1['accuracy']):.4f}" if l1 else "N/A"
        html += f"<tr><td><b>L2 (LLM 1)</b></td><td>{l2['accuracy']:.4f}</td><td>{l2['macro_f1']:.4f}</td><td>{gain}</td></tr>"
    else:
        html += "<tr><td><b>L2 (LLM 1)</b></td><td colspan='3'>❌ FAILED</td></tr>"
    
    if l3:
        gain = f"+{(l3['accuracy'] - l1['accuracy']):.4f}" if l1 else "N/A"
        html += f"<tr><td><b>L3 (LLM 3)</b></td><td>{l3['accuracy']:.4f}</td><td>{l3['macro_f1']:.4f}</td><td>{gain}</td></tr>"
    else:
        html += "<tr><td><b>L3 (LLM 3)</b></td><td colspan='3'>❌ FAILED</td></tr>"
    
    html += "</table><br>"
    display(HTML(html))

## 🏃 Run All Experiments

In [ ]:
# Group configs — pattern: exp_{dataset}_mpnet_{mode}.yaml
# Modes: name_only (L1), l2_anchored (L2), l3_anchored (L3)
config_dir = Path("experiments")
configs_by_group = {}

MODES = {
    "name_only": "name_only",
    "l2_anchored": "l2_anchored",
    "l3_anchored": "l3_anchored",
}

for config_path in sorted(config_dir.glob("exp_*_mpnet_*.yaml")):
    stem = config_path.stem  # e.g. exp_ag_news_mpnet_name_only
    
    # Determine mode
    mode = None
    for m in MODES:
        if stem.endswith(f"_{m}"):
            mode = m
            break
    if mode is None:
        continue
    
    # Remove exp_ prefix and _{mode} suffix to get dataset
    # stem = exp_{dataset}_mpnet_{mode}
    inner = stem[4:]  # remove 'exp_'
    inner = inner[:-(len(mode) + 1)]  # remove '_{mode}'
    # inner is now {dataset}_mpnet — strip _mpnet
    if inner.endswith("_mpnet"):
        dataset = inner[:-6]
    else:
        dataset = inner
    
    # Apply filters
    if FILTER_DATASET and FILTER_DATASET not in dataset:
        continue
    
    if dataset not in configs_by_group:
        configs_by_group[dataset] = {}
    configs_by_group[dataset][mode] = config_path

print(f"Found {len(configs_by_group)} dataset groups")
total_experiments = sum(len(modes) for modes in configs_by_group.values())
print(f"Total experiments: {total_experiments}")

for name, modes in sorted(configs_by_group.items()):
    print(f"  {name}: {list(modes.keys())}")

In [ ]:
from tqdm.notebook import tqdm
import time

print("="*80)
print("STARTING LABEL SEMANTICS EXPERIMENTS")
print("="*80)

all_results = []
start_time = time.time()

for idx, (dataset, configs) in enumerate(tqdm(sorted(configs_by_group.items()), desc="Datasets")):
    print(f"\n{'#'*80}")
    print(f"# {idx+1}/{len(configs_by_group)}: {dataset}")
    print(f"{'#'*80}")
    
    # Run L1 (name_only), L2 (l2_anchored), L3 (l3_anchored)
    l1_metrics = run_experiment(configs.get('name_only'), SKIP_EXISTING) if 'name_only' in configs else None
    l2_metrics = run_experiment(configs.get('l2_anchored'), SKIP_EXISTING) if 'l2_anchored' in configs else None
    l3_metrics = run_experiment(configs.get('l3_anchored'), SKIP_EXISTING) if 'l3_anchored' in configs else None
    
    # Store results
    all_results.append({
        'dataset': dataset,
        'l1_acc': l1_metrics['accuracy'] if l1_metrics else None,
        'l2_acc': l2_metrics['accuracy'] if l2_metrics else None,
        'l3_acc': l3_metrics['accuracy'] if l3_metrics else None,
        'l1_f1': l1_metrics['macro_f1'] if l1_metrics else None,
        'l2_f1': l2_metrics['macro_f1'] if l2_metrics else None,
        'l3_f1': l3_metrics['macro_f1'] if l3_metrics else None,
    })
    
    if SHOW_PROGRESS:
        show_comparison(l1_metrics, l2_metrics, l3_metrics, dataset, "mpnet")

elapsed = time.time() - start_time
print(f"\n{'='*80}")
print(f"ALL EXPERIMENTS COMPLETED!")
print(f"Time: {elapsed/60:.1f} minutes")
print(f"{'='*80}")

## 📊 Results Analysis

In [ ]:
import pandas as pd

# Create DataFrame
df_results = pd.DataFrame(all_results)

# Calculate gains
df_results['l2_gain'] = df_results['l2_acc'] - df_results['l1_acc']
df_results['l3_gain'] = df_results['l3_acc'] - df_results['l1_acc']
df_results['l3_vs_l2'] = df_results['l3_acc'] - df_results['l2_acc']

# Save
df_results.to_csv("results/label_semantics_l1_l2_l3.csv", index=False)
print("✅ Saved: results/label_semantics_l1_l2_l3.csv")

df_results.head(10)

In [ ]:
# Overall statistics
print("="*80)
print("OVERALL STATISTICS")
print("="*80)

print("\nMean Accuracy:")
print(f"L1 (name_only): {df_results['l1_acc'].mean():.4f}")
print(f"L2 (LLM 1):     {df_results['l2_acc'].mean():.4f}")
print(f"L3 (LLM 3):     {df_results['l3_acc'].mean():.4f}")

print("\nMean Gains:")
print(f"L2 vs L1: {df_results['l2_gain'].mean():+.4f} ({df_results['l2_gain'].mean()*100:+.2f}%)")
print(f"L3 vs L1: {df_results['l3_gain'].mean():+.4f} ({df_results['l3_gain'].mean()*100:+.2f}%)")
print(f"L3 vs L2: {df_results['l3_vs_l2'].mean():+.4f} ({df_results['l3_vs_l2'].mean()*100:+.2f}%)")

In [ ]:
# Per-dataset analysis
print("="*80)
print("PER-DATASET ANALYSIS")
print("="*80)

dataset_stats = df_results.groupby('dataset')[['l1_acc', 'l2_acc', 'l3_acc', 'l2_gain', 'l3_gain']].mean()
dataset_stats = dataset_stats.sort_values('l3_gain', ascending=False)
print(dataset_stats.to_string())

## 📈 Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Overall comparison (Macro F1)
ax = axes[0]
means_f1 = [df_results['l1_f1'].mean(), df_results['l2_f1'].mean(), df_results['l3_f1'].mean()]
bars = ax.bar(['L1 (name_only)', 'L2 (anchored)', 'L3 (anchored)'], means_f1,
              color=['#e74c3c', '#3498db', '#2ecc71'])
ax.set_ylabel('Macro F1')
ax.set_title('Mean Macro F1 by Label Level\n(9 datasets, mpnet)')
ax.set_ylim([0, 1])
for i, v in enumerate(means_f1):
    ax.text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')

# 2. Per-dataset F1 comparison
ax = axes[1]
dataset_means = df_results.set_index('dataset')[['l1_f1', 'l2_f1', 'l3_f1']].sort_values('l3_f1')
dataset_means.columns = ['L1', 'L2', 'L3']
dataset_means.plot(kind='barh', ax=ax, color=['#e74c3c', '#3498db', '#2ecc71'])
ax.set_xlabel('Macro F1')
ax.set_title('Macro F1 by Dataset and Label Level')

# 3. Gains over L1
ax = axes[2]
df_results_sorted = df_results.set_index('dataset').sort_values('l3_f1')
x = range(len(df_results_sorted))
ax.bar([i - 0.2 for i in x], df_results_sorted['l2_f1'] - df_results_sorted['l1_f1'],
       width=0.4, label='L2 vs L1', color='#3498db', alpha=0.8)
ax.bar([i + 0.2 for i in x], df_results_sorted['l3_f1'] - df_results_sorted['l1_f1'],
       width=0.4, label='L3 vs L1', color='#2ecc71', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(df_results_sorted.index, rotation=45, ha='right', fontsize=8)
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('F1 Gain')
ax.set_title('F1 Gain over L1 (name_only)')
ax.legend()

plt.tight_layout()
plt.savefig('results/label_semantics_analysis.png', dpi=300, bbox_inches='tight')
print("✅ Saved: results/label_semantics_analysis.png")
plt.show()

## 💾 Download Results

In [ ]:
# Zip results
!zip -r label_semantics_results.zip results/raw/label_semantics/ results/label_semantics_l1_l2_l3.csv results/label_semantics_analysis.png

from google.colab import files
files.download('label_semantics_results.zip')

print("✅ Results downloaded!")

## ✅ Done!

**Results:**
- `results/raw/label_semantics/` - All metrics and predictions (27 experiments)
- `results/label_semantics_l1_l2_l3.csv` - Summary table
- `results/label_semantics_analysis.png` - Visualizations

**Configs Used:**
- `experiments/` - 27 YAML configs (exp_*_mpnet_*.yaml)
- Modes: `name_only` (L1), `l2_anchored` (L2), `l3_anchored` (L3)
- Model: `sentence-transformers/all-mpnet-base-v2`